In [2]:
print('hwllow')

hwllow


In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
import os
import glob
import chromadb
from chromadb.config import Settings
import sys
sys.path.append(r"C:\Users\rauna\projects\My Projects\Drug Chatbot")
sys.path.append(r"C:\Users\rauna\projects\My Projects\Drug Chatbot\vector_embedding_methods")

c:\Users\rauna\projects\My Projects\Drug Chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import gradio as gr

In [5]:
# MODEL_NAME = "all-MiniLM-L6-v2"
# DB_PATH = r"C:\Users\rauna\projects\llm_engineering\My Projects\Drug Chatbot\ChromaDB\chroma_db"
# COLLECTION_NAME = "document_embeddings"

from utils_embedding import embedding_model,DB_PATH,FULL_FILE_COLLECTION_NAME,chroma_client
from utils import call_llm,MODEL

In [ ]:
# model = SentenceTransformer('all-MiniLM-L6-v2')

In [6]:
# chroma_client = chromadb.PersistentClient(path=DB_PATH)
full_doc_collection = chroma_client.get_collection(FULL_FILE_COLLECTION_NAME)

In [7]:
def retrieve_documents_paracetamol(message: str, n_results: int = 3):
    system_message = '''You extract paracetamol related messages from the user message and retrun only 'paracetamol' related questions.
                        Only return the extracted message without any explaniations.
                        Igone any other medicine name that is mentioned.'''
    paracetamol_message = call_llm(MODEL,system_message,message)
    print(paracetamol_message)
    message_embedding = embedding_model.encode(paracetamol_message).tolist()

    results = full_doc_collection.query(
        query_embeddings=[message_embedding],
        n_results=n_results
    )
    return results

In [8]:
def retrieve_documents_insulin(message: str, n_results: int = 5):
    system_message = '''You extract Insulin related messages from the user message and retrun only 'Insulin' related questions.
                        Only return the extracted message without any explaniations.
                        Igone any other medicine name that is mentioned.'''
    insulin_message = call_llm(MODEL,system_message,message)
    print(insulin_message)
    message_embedding = embedding_model.encode(insulin_message).tolist()

    results = full_doc_collection.query(
        query_embeddings=[message_embedding],
        n_results=n_results
    )
    return results

In [9]:
def build_context(documents) -> str:
    docs = documents["documents"][0]      
    metadatas = documents["metadatas"][0]  # list of metadata dicts

    context_blocks = []
    for doc, meta in zip(docs, metadatas):
        filename = meta.get("filename", "unknown_source.txt")
        person_name = meta.get("person", "Unknown Person")
        context_blocks.append(f"Source: {person_name}\n{filename}\n{doc}")

    context = "\n\n---\n\n".join(context_blocks)
    return context

In [13]:
result = retrieve_documents_paracetamol('who is paracetamol')
docs = result["documents"][0]
metadatas = result["metadatas"][0]
display(result["documents"][0] )
display(result["metadatas"][0] )

Who is paracetamol?


['Topic: Weaknesses and areas for growth\n\n## Paracetamol: Strengths and Areas for Growth\n\nParacetamol, a familiar name in many households, is a highly effective and widely used analgesic and antipyretic. Its strengths lie in its accessibility, affordability, and a generally favorable safety profile when used as directed. However, like any pharmaceutical, Paracetamol isn\'t without its limitations and areas where further understanding and cautious application are crucial.\n\n### Key Strengths:\n\n*   **Ubiquitous Accessibility and Affordability:** One of Paracetamol\'s most significant strengths is its widespread availability. It\'s an over-the-counter medication found in virtually every pharmacy, supermarket, and even convenience store. This ease of access makes it a go-to for immediate relief from common ailments. Furthermore, its affordability ensures that it\'s a viable option for a broad spectrum of the population, regardless of socioeconomic status.\n\n*   **Effective Pain and

[{'person': 'Paracetamol', 'filename': 'Paracetamol_file_11.txt'},
 {'person': 'Paracetamol', 'filename': 'Paracetamol_file_08.txt'},
 {'filename': 'Paracetamol_file_17.txt', 'person': 'Paracetamol'},
 {'person': 'Paracetamol', 'filename': 'Paracetamol_file_06.txt'},
 {'person': 'Paracetamol', 'filename': 'Paracetamol_file_07.txt'}]

In [17]:
context_blocks = []
for doc, meta in zip(docs, metadatas):
    filename = meta.get("filename", "unknown_source.txt")
    person_name = meta.get("person", "Unknown Person")
    context_blocks.append(f"Source: Person - {person_name}\nfile_name : {filename}\n{doc}")
display(context_blocks)

['Source: Person - Paracetamol\nfile_name : Paracetamol_file_11.txt\nTopic: Weaknesses and areas for growth\n\n## Paracetamol: Strengths and Areas for Growth\n\nParacetamol, a familiar name in many households, is a highly effective and widely used analgesic and antipyretic. Its strengths lie in its accessibility, affordability, and a generally favorable safety profile when used as directed. However, like any pharmaceutical, Paracetamol isn\'t without its limitations and areas where further understanding and cautious application are crucial.\n\n### Key Strengths:\n\n*   **Ubiquitous Accessibility and Affordability:** One of Paracetamol\'s most significant strengths is its widespread availability. It\'s an over-the-counter medication found in virtually every pharmacy, supermarket, and even convenience store. This ease of access makes it a go-to for immediate relief from common ailments. Furthermore, its affordability ensures that it\'s a viable option for a broad spectrum of the populati

In [19]:
build_context(result)
display(build_context(result))

'Source: Paracetamol\nParacetamol_file_11.txt\nTopic: Weaknesses and areas for growth\n\n## Paracetamol: Strengths and Areas for Growth\n\nParacetamol, a familiar name in many households, is a highly effective and widely used analgesic and antipyretic. Its strengths lie in its accessibility, affordability, and a generally favorable safety profile when used as directed. However, like any pharmaceutical, Paracetamol isn\'t without its limitations and areas where further understanding and cautious application are crucial.\n\n### Key Strengths:\n\n*   **Ubiquitous Accessibility and Affordability:** One of Paracetamol\'s most significant strengths is its widespread availability. It\'s an over-the-counter medication found in virtually every pharmacy, supermarket, and even convenience store. This ease of access makes it a go-to for immediate relief from common ailments. Furthermore, its affordability ensures that it\'s a viable option for a broad spectrum of the population, regardless of soci

In [10]:
from dotenv import load_dotenv
import os
from openai import OpenAI
from IPython.display import Markdown, display
import gradio as gr
import ast

load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

openrouter_url = "https://openrouter.ai/api/v1"

openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)

MODEL = 'openai/gpt-4o-2024-11-20'


OpenRouter API Key exists and begins sk-


In [11]:
def call_llm(model, system_message, user_message):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
    response = openrouter.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content


def call_llm_with_history(model, system_message, history, user_message):
    messages = [{"role": "system", "content": system_message}] + history + [
        {"role": "user", "content": user_message}
    ]
    response = openrouter.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content


def safe_eval(text):
    return ast.literal_eval(text)

In [12]:
def rag_answer_paracetamol(message):
    system_message = '''You extract paracetamol related messages from the user message and retrun only 'paracetamol' related questions.
                        Only return the compleate and extracted paracaetamol related message without any explaniations.
                        Ignore any other medicine name that is mentioned.'''
    paracetamol_message = call_llm(MODEL,system_message,message)    
 
    results = retrieve_documents_paracetamol(paracetamol_message)
    context = build_context(results)

    system_message = f"""You are a helpful drug information assistant.

Use ONLY the following context to answer the question. 
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {paracetamol_message}

Answer:"""

    answer = call_llm(MODEL, system_message, paracetamol_message)
    print(system_message)
    return answer

In [13]:
def rag_answer_insulin(message):
    system_message = '''You extract insulin related messages from the user message and retrun only 'insulin' related questions.
                        Only return the extracted message without any explaniations.
                        Igone any other medicine name that is mentioned.'''
    insulin_message = call_llm(MODEL,system_message,message)    
 
    results = retrieve_documents_insulin(insulin_message)
    context = build_context(results)

    system_message = f"""You are a helpful drug information assistant.

Use ONLY the following context to answer the question. 
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {insulin_message}

Answer:"""
    print(system_message)
    answer = call_llm(MODEL, system_message, insulin_message)
    return answer

In [14]:
def cat_agent(message):
    system_message = '''You are a helpful assistant that catergorizes the given message into two catergories.
                        Category 1 : General Query
                        Category 2 : Drug Information
                        If the message contains any drug or medicine name then put it into Category 2 : Drug Information
                        Return only the category decision of the whole message.
    '''
    category = call_llm(MODEL, system_message, message)
    print(category)
    return category

def reply_normal_agent(message,history):
    system_message = 'You are a helpful assistant that can answer questions and help with tasks.'
    response = call_llm_with_history(MODEL, system_message,history, message)
    return response


def paracetamol_agent(message):
    print('paracetamol_agent is running')
    # system_message = '''You are a helpful assistant that answers questions only related to paracetamol and nothing else.
    #                     If any other drug is mentioned then completly ignore its mention and do not talk about it.
    #                     Return results in Dict format where the keys are different medicine names
    #                     Return ONLY a Python dict.
    #                     Do NOT wrap the response in backticks.
    #                     Do NOT use JSON.
    #                     Do NOT add explanations.
    #                     Example : user message : what is paracetamol. what is insulin
    #                               assistant message : {'paracetamol' : 'details about paracetamol',
    #                                                     'insulin' : 'details about insulin'}'''
    raw_paractamol_output = rag_answer_paracetamol(message)
    # paracetamol = safe_eval(raw_paractamol_output)
    # return paracetamol['paracetamol']
    return raw_paractamol_output


def insulin_agent(message):
    print('insulin_agent is running')
    # system_message = '''You are a helpful assistant that answers questions only related to insulin and nothing else.
    #                     If any other drug is mentioned then completly ignore its mention and do not talk about it.
    #                     Return results in Dict format where the keys are different medicine names
    #                     Return ONLY a Python dict.
    #                     Do NOT wrap the response in backticks.
    #                     Do NOT use JSON.
    #                     Do NOT add explanations.
    #                     Example : user message : what is paracetamol. what is insulin
    #                               assistant message : {'paracetamol' : 'details about paracetamol',
    #                                                     'insulin' : 'details about insulin'}'''
    raw_insulin_output = rag_answer_insulin(message)
    # insulin = safe_eval(raw_insulin_output)
    # return insulin['insulin']
    return raw_insulin_output

def drug_reply(message):
    system_message = '''You are a helpful assistant that picks out all the drug names in a query and gives a list of the drug names present in the query.
                        Return only python list
                        Fix the typos as well if you find any'''
    drugs_raw = call_llm(MODEL, system_message, message)
    drugs = safe_eval(drugs_raw)
    print(drugs)

    results = []

    for drug in drugs:
        if drug.lower() == 'paracetamol':
            results.append(paracetamol_agent(message))

        elif drug.lower() == 'insulin' :
            results.append(insulin_agent(message)) 
        
        else :
            results.append(f"Invalid drug {drug}. I only answer questions related to paracetamol and insulin")

    return results



In [15]:
def orch(message,history) :
    catergory = cat_agent(message)
    if 'General Query' in catergory:
        return reply_normal_agent(message,history)
    elif 'Drug Information' in catergory:
        return drug_reply(message)
    else :
        return "Sorry, I could not categorize your question. Please try rephrasing it."

In [ ]:
# gr.ChatInterface(fn=orch, type="messages").launch()
gr.ChatInterface(fn=orch).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Category 2 : Drug Information
['Paracetamol']
paracetamol_agent is running
What is the gender of paracetamol?
You are a helpful drug information assistant.

Use ONLY the following context to answer the question. 
If the answer is not in the context, say you don't know.

Context:
Source: Paracetamol
Paracetamol_file_11.txt
Topic: Weaknesses and areas for growth

## Paracetamol: Strengths and Areas for Growth

Paracetamol, a familiar name in many households, is a highly effective and widely used analgesic and antipyretic. Its strengths lie in its accessibility, affordability, and a generally favorable safety profile when used as directed. However, like any pharmaceutical, Paracetamol isn't without its limitations and areas where further understanding and cautious application are crucial.

### Key Strengths:

*   **Ubiquitous Accessibility and Affordability:** One of Paracetamol's most significant strengths is its widespread availability. It's an over-the-counter medication found in virtu